<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/AI_Powered_Company_Intelligence_Database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Company Intelligence & Search Engine
This notebook downloads company data, enriches it using AI (Gemini), and provides a searchable interface using DuckDB.

In [1]:
!pip install -q pandas duckdb google-generativeai pyarrow requests

In [2]:
import pandas as pd
import duckdb
import google.generativeai as genai
import requests
import os
import time
from google.colab import userdata

# Configuration
STORAGE_PATH = 'company_data.parquet'
BATCH_SIZE = 1000

# AI Setup
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel('gemini-1.5-flash')
except Exception as e:
    print(f"Warning: AI setup failed. Ensure GOOGLE_API_KEY is in secrets. Error: {e}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 1. Data Acquisition
We will download a public dataset of companies. For this demonstration, we'll use a sample dataset.

In [6]:
def download_data():
    # Primary URL for company data
    url = 'https://raw.githubusercontent.com/datasets/company-list/master/data/companies.csv'
    try:
        df = pd.read_csv(url)
        df = df.rename(columns={'Name': 'company_name', 'Domain': 'website', 'Industry': 'industry', 'Country': 'location'})
    except Exception as e:
        print(f"Could not download dataset: {e}. Generating sample data instead.")
        # Fallback sample data
        data = {
            'company_name': ['Google', 'Microsoft', 'Apple', 'Tesla', 'Meta', 'Amazon', 'Netflix', 'Nvidia', 'Adobe', 'Salesforce'],
            'website': ['google.com', 'microsoft.com', 'apple.com', 'tesla.com', 'meta.com', 'amazon.com', 'netflix.com', 'nvidia.com', 'adobe.com', 'salesforce.com'],
            'industry': ['Technology', 'Software', 'Hardware', 'Automotive', 'Social Media', 'E-commerce', 'Entertainment', 'Semiconductors', 'Software', 'Cloud Software'],
            'location': ['USA', 'USA', 'USA', 'USA', 'USA', 'USA', 'USA', 'USA', 'USA', 'USA']
        }
        df = pd.DataFrame(data)

    return df[['company_name', 'website', 'industry', 'location']]

raw_df = download_data()
display(raw_df.head())

Could not download dataset: HTTP Error 404: Not Found. Generating sample data instead.


,company_name,website,industry,location
0,Google,google.com,Technology,USA
1,Microsoft,microsoft.com,Software,USA
2,Apple,apple.com,Hardware,USA
3,Tesla,tesla.com,Automotive,USA
4,Meta,meta.com,Social Media,USA


## 2. AI Enrichment & Incremental Processing
We process records in batches and save progress to Parquet to support resumes.

In [7]:
def enrich_with_ai(row):
    prompt = f"""Generate a brief JSON-style summary for {row['company_name']} in industry {row['industry']}:
    - summary: 1 sentence
    - icp: ideal customer profile
    - competitors: top 3 competitors"""
    try:
        # Only attempt if model was successfully initialized
        if 'model' in globals():
            response = model.generate_content(prompt)
            return response.text
        else:
            return "AI logic skipped: No API Key"
    except Exception as e:
        return f"AI Error: {str(e)}"

def process_incrementally(df):
    if os.path.exists(STORAGE_PATH):
        processed_df = pd.read_parquet(STORAGE_PATH)
        processed_names = set(processed_df['company_name'])
    else:
        processed_df = pd.DataFrame()
        processed_names = set()

    new_records = []
    count = 0

    for _, row in df.iterrows():
        if row['company_name'] in processed_names: continue

        row_data = row.to_dict()
        row_data['ai_info'] = enrich_with_ai(row)
        new_records.append(row_data)
        count += 1

        if count % BATCH_SIZE == 0:
            batch_df = pd.DataFrame(new_records)
            processed_df = pd.concat([processed_df, batch_df], ignore_index=True)
            processed_df.to_parquet(STORAGE_PATH)
            new_records = []
            print(f"Saved batch: {count} records")

    if new_records:
        batch_df = pd.DataFrame(new_records)
        processed_df = pd.concat([processed_df, batch_df], ignore_index=True)
        processed_df.to_parquet(STORAGE_PATH)

    return processed_df

# Ensure raw_df exists before processing
if 'raw_df' in globals():
    final_df = process_incrementally(raw_df.head(10))
    print("Processing complete.")

Processing complete.


## 3. Search Interface with DuckDB
We use DuckDB to query the Parquet file directly.

In [8]:
def search_companies(query):
    import os
    if not os.path.exists(STORAGE_PATH):
        return "Error: Data file not found. Please run the processing cell first."

    con = duckdb.connect()
    try:
        # Using an f-string to inject the path directly for clarity
        query_sql = f"""
            SELECT * FROM read_parquet('{STORAGE_PATH}')
            WHERE company_name ILIKE '%{query}%'
            OR industry ILIKE '%{query}%'
        """
        result = con.execute(query_sql).df()
        return result
    except Exception as e:
        return f"Search Error: {e}"
    finally:
        con.close()

# Example Search - searching for 'Tech' in names or industries
search_results = search_companies('Tech')
display(search_results)

,company_name,website,industry,location,ai_info
0,Google,google.com,Technology,USA,AI logic skipped: No API Key
